In [4]:
### ChromaDB connection test
import os
from dotenv import load_dotenv

load_dotenv()
from ingest.config import chroma_host, chroma_port, embedding_model
import chromadb

host, port = "127.0.0.1", 8000
c_client = chromadb.HttpClient(host=host, port=port)
print(f"ChromaDB connected with {host}:{port} and embedding model {embedding_model()}")


def get_collection(client, collection_name="chunks"):
    return client.get_or_create_collection(name=collection_name)

def get_user_chunks(client):
    res = get_collection(client).get(include=["metadatas"])
    with open("user_chunks.json", "w") as f:
        import json
        json.dump(res, f)
    return res


def create_voyage():
    import voyageai
    
    client = voyageai.Client(api_key=os.environ["VOYAGE_API_KEY"])  # from DocuLyze/.env (repo root)
    model = embedding_model()
    print(f"VoyageAI connected with embedding model {model}")
    def embed_function(texts: list[str], token_size: list[int], input_type:str) -> list[list[float]]:
    # Placeholder embedding function; replace with actual embedding logic
        if sum(token_size) > 1000000: # voyage's 32k token context window
            print(f"WARNING: embedding {len(texts)} chunks with total token size {sum(token_size)} exceeds 1M maximum limit; "
                "Voyage fails",
            )
            return
        return client.embed(texts, model=model, input_type=input_type).embeddings 
    return embed_function

def write_user_chunks(client, embed_fn, uid: str, doc_id: str, chunks: list[str]) -> None:
    coll = client.get_or_create_collection("chunks")
    coll.delete(where={"$and": [{"userId": uid}, {"docId": doc_id}]})
    contents = [c.content for c in chunks]
    token_sizes = [c.tokens for c in chunks]
    embeddings = embed_fn(contents, token_size=token_sizes, input_type="document")
    print(f"sample created embedding {embeddings[0] if embeddings else None}")

    coll.add(
        ids=[f"{uid}/{doc_id}/{idx}" for idx, _ in enumerate(chunks)],
        embeddings=embeddings,
        documents=contents,
        metadatas=[
            {
                "userId": uid,
                "docId": doc_id,
                "chunk_index": idx,
                "token_count": c.tokens,
                "embedding_model": embedding_model(),
                "breadcrumb": c.breadcrumb(),
            }
            for idx, c in enumerate(chunks)
        ],
    )

def query(client, embed_function, user_query: str, top_k: int = 3):
    coll = client.get_collection("chunks")
    embeddings = embed_function([user_query], token_size=[len(user_query.split())], input_type="query")
    res = coll.query(
        query_embeddings = embeddings,
        n_results=top_k,
        include=["metadatas", "distances", "documents"],

    )
    documents = res["documents"][0]
    metadatas = res["metadatas"][0]
    distances = res["distances"][0]
    return metadatas, distances, documents



ChromaDB connected with 127.0.0.1:8000 and embedding model voyage-3.5-lite


In [ ]:
"""Walk an input directory, parse each supported file to markdown, and split into chunks."""

import sys
import os
from pathlib import Path

from ingest.parsers import PARSERS, parser_for
from ingest.chunker import chunk_markdown


src_dir = Path("C:\\Users\\jw300\\DocuLyze\\ingest-worker\\experimental_results\\source_files")

EXT_TO_TYPE = {
    ".pdf": "application/pdf",
    ".docx": "application/vnd.openxmlformats-officedocument.wordprocessingml.document",
    ".txt": "text/plain",
    ".md": "text/markdown",
    ".html": "text/html",
    ".css": "text/css",
    ".json": "application/json",
    ".js": "application/javascript",
    ".ts": "application/typescript",
    ".xml": "application/xml",
    ".py": "application/x-python",
}


def ext_to_content_type(name: str) -> str | None:
    suffix = Path(name).suffix.lower()
    return EXT_TO_TYPE.get(suffix)


def parse_and_split(src: Path) -> list:
    mime = ext_to_content_type(src.name)
    if not mime:
        print(f"  ignored (unrecognized extension): {src.name}")
        return []

    try:
        key = parser_for(mime)
    except KeyError:
        print(f"  ignored (no parser registered for {mime}): {src.name}")
        return []

    raw = src.read_bytes()
    if PARSERS[key] == PARSERS["pdf"]:
        md_text = PARSERS[key].parse(raw, max_pages=20)
    else:
        md_text = PARSERS[key].parse(raw)

    if not md_text.strip():
        print(f"  ignored (parser returned nothing): {src.name}")
        return []
    if not os.path.exists(os.path.join(src_dir, "converted_md", f"{src.stem}.md")):
        # os.makedirs(os.path.join(src_dir, "converted_md"), exist_ok=True)
        with open(os.path.join(src_dir, "converted_md", f"{src.stem}.md"), "w", encoding="utf-8") as f:
            f.write(md_text)
    pieces = chunk_markdown(md_text)

    # one folder per source file under experimental_results/chunks/: chunk_<filename>/
    piece_dir = src_dir.parent / "chunks" / f"chunk_{src.stem}"
    piece_dir.mkdir(parents=True, exist_ok=True)
    for stale in piece_dir.glob("chunk_*.md"):  # re-runs must not leave old chunks behind
        stale.unlink()
    for piece in pieces:
        breadcrumb = f"<!-- {piece.breadcrumb()} -->\n\n" if piece.breadcrumb() else ""
        (piece_dir / f"chunk_{piece.metadata['chunk_index']:03d}.md").write_text(
            breadcrumb + piece.content, encoding="utf-8"
        )

    return pieces




### Test Ingest Worker
The cell below runs the same chunking + embedding pipeline as the ingest worker to verify that the process works correctly with sample data. Every file within '../experimental_results/chunks/' will be parsed, chunked, and embedded.

In [10]:
### chunkers
from ingest.chunker import chunk_markdown


# c_client.delete_collection("chunks")
embed = create_voyage()
entries = sorted(f for f in src_dir.iterdir() if f.is_file())
if not entries:
    print(f"No files in {src_dir}")
    sys.exit(1)

chunk_count = 0
for entry in entries:
    print(f"Processing: {entry.name}")
    pieces = parse_and_split(entry)
    if pieces:
        token_sum = sum(p.tokens for p in pieces)
        print(f"  {len(pieces)} chunks, {token_sum} tokens")
        chunk_count += len(pieces)
    # write_user_chunks(c_client, embed, "tmwang", f"f123123{chunk_count}", pieces)
    

print(f"\nFinished: {len(entries)} files, {chunk_count} chunks")


VoyageAI connected with embedding model voyage-3.5-lite
Processing: 2025_AnnualReport.docx


KeyError: 'application/pdf'

### Recall@k + NDCG evaluation test

In [ ]:
import numpy as np


def ranked_list_to_relevance(ordered_chunks: list[int]) -> dict[int, int]:
    """Convert a most-to-least-relevant ordered list into graded relevance scores.

    [13, 0] → {13: 2, 0: 1}  (first element = highest score = len)
    """
    n = len(ordered_chunks)
    return {chunk_id: n - i for i, chunk_id in enumerate(ordered_chunks)}


def recall_k(retrieved_ids: list[int], relevant_ids: dict[int, int], k: int) -> float:
    retrieved_at_k = retrieved_ids[:k]
    relevant_retrieved = [id for id in retrieved_at_k if id in relevant_ids]
    return len(relevant_retrieved) / len(relevant_ids) if relevant_ids else 0.0


def ndcg_k(retrieved_ids: list[int], relevant_ids: dict[int, int], k: int) -> float:
    retrieved_at_k = retrieved_ids[:k]
    dcg = 0.0
    for i, id in enumerate(retrieved_at_k):
        if id in relevant_ids:
            dcg += relevant_ids[id] / np.log2(i + 2)

    sorted_relevances = sorted(relevant_ids.values(), reverse=True)
    ideal_dcg = 0.0
    for i in range(min(len(sorted_relevances), k)):
        ideal_dcg += sorted_relevances[i] / np.log2(i + 2)
    return dcg / ideal_dcg if ideal_dcg > 0 else 0.0

### Query Test


In [ ]:
import json
import os
import csv

import openpyxl

json_path = os.path.join(src_dir.parent, "sample_q&a", "google_questions.json")
with open(json_path, "r") as f:
    google_data = json.load(f)

K_VALUES = [1, 3, 5]
results = []
google_easy_questions = [q for q in google_data["questions"] if q["difficulty"] == "easy"]
google_medium_questions = [q for q in google_data["questions"] if q["difficulty"] == "medium"]
google_hard_questions = [q for q in google_data["questions"] if q["difficulty"] == "hard"]
for question in google_data["questions"]:
    relevance = ranked_list_to_relevance(question["relevant_chunks"])
    metadatas, distances, documents = query(c_client, embed, question["question"], top_k=max(K_VALUES))
    retrieved_ids = [m["chunk_index"] for m in metadatas]

    question["retrieved_contexts"] = documents
    question["retrieved_chunk_indices"] = retrieved_ids

    row = {
        "id": question["id"],
        "question": question["question"],
        "ground_truth_answer": question["ground_truth_answer"],
        "difficulty": question["difficulty"],
        "type": question["type"],
        "num_relevant": len(question["relevant_chunks"]),
        "relevant_chunks": str(question["relevant_chunks"]),
        "retrieved_chunks": str(retrieved_ids),
    }
    for k in K_VALUES:
        row[f"recall@{k}"] = round(recall_k(retrieved_ids, relevance, k), 4)
        row[f"ndcg@{k}"] = round(ndcg_k(retrieved_ids, relevance, k), 4)

    results.append(row)
    print(f"{question['id']:16s} recall@3={row['recall@3']:.3f}  ndcg@3={row['ndcg@3']:.3f} recall@5={row['recall@5']:.3f}  ndcg@5={row['ndcg@5']:.3f}  retrieved={retrieved_ids[:5]}")

with open(json_path, "w") as f:
    json.dump(google_data, f, indent=2)
print(f"\nWrote retrieved_contexts back to {json_path}")

csv_path = os.path.join(src_dir.parent, "sample_q&a", "manual_eval_results.xlsx")
fieldnames = list(results[0].keys())
wb = openpyxl.Workbook()
ws = wb.active
ws.append(fieldnames)
for row in results:
    ws.append([row[field] for field in fieldnames])
wb.save(csv_path)

print(f"Results written to {csv_path}")
print(f"Questions: {len(results)}")
for k in K_VALUES:
    avg_recall = np.mean([r[f"recall@{k}"] for r in results])
    avg_ndcg = np.mean([r[f"ndcg@{k}"] for r in results])
    print(f"  Mean recall@{k}: {avg_recall:.4f}   Mean NDCG@{k}: {avg_ndcg:.4f}")

### RAGAS Evaluation

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

from openai import OpenAI
from ragas import evaluate, EvaluationDataset, SingleTurnSample
from ragas.llms import llm_factory
from ragas.metrics import LLMContextPrecisionWithReference, LLMContextRecall, NonLLMContextRecall

judge_llm = llm_factory(
    "gpt-4o-mini",
    client=OpenAI(api_key=os.environ["OPENAI_API_KEY"]),  # from DocuLyze/.env (repo root)
)

samples = []
map = {"easy": google_easy_questions,
       "medium": google_medium_questions,
       "hard": google_hard_questions,
       "all": google_data["questions"]
       }
difficulty = "hard"
# for q in google_data["questions"]:
for q in map["all"]:
# for q in google_easy_questions:
    contexts = q.get("retrieved_contexts")
    if not contexts:
        continue
    samples.append(SingleTurnSample(
        user_input=q["question"],
        retrieved_contexts=contexts,
        reference_contexts=q["relevant_context"],
        reference=q["ground_truth_answer"],
    ))

dataset = EvaluationDataset(samples=samples)

result = evaluate(
    dataset=dataset,
    metrics=[LLMContextPrecisionWithReference(), LLMContextRecall(), NonLLMContextRecall()],
    llm=judge_llm,
)
print(result)
result.to_pandas().to_excel(os.path.join(src_dir.parent, "sample_q&a", f"RAGAS_google_{difficulty}_eval_results.xlsx"))


In [ ]:
result.to_pandas()


## Test for Hybrid Retrieval with GPT-4.1 NER

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model="gpt-4",
    api_key=os.environ["OPENAI_API_KEY"],  # from DocuLyze/.env (repo root)
    temperature=0
)
### problematic
q_bank = ["What was Alphabet's free cash flow in Q1 2026 and for the trailing twelve months ending Q1 2026?",
          "What was Alphabet's total revenue from the United States in Q1 2026, and how did it compare year-over-year?",
          "How did YouTube ads revenue perform in Q1 2026 compared to Q1 2025?",
          "Where would one contact the investors and media of Alphabet?",
          "What evidence from the Q1 2026 earnings release supports the claim that Alphabet is heavily investing in AI, and what returns are those investments generating?",
          "Summarize Alphabet's overall financial performance in Q1 2026, covering revenue, profitability, and cash generation."
        ]
question = q_bank[1]
_, _, retrieved_contexts = query(c_client, embed, question, top_k=5)
print(question)
print()
context = "\n\n---\n\n".join(retrieved_contexts)
response = llm.invoke([
    SystemMessage(
        content=(
            "Answer the following question based only on the provided context. "
            # "If the context does not contain the answer, respond with 'I don't know.'"
        )
    ),
    HumanMessage(content=f"Context:\n{context}\n\nQuestion:\n{question}"),

])

print(response.content)

### GLiNER NER Extraction Test
Test zero-shot NER on the full Alphabet earnings document using GLiNER.
Extracts: PERSON, ORGANIZATION, LOCATION, DATE, CONCEPT/TECHNICAL TERM.
EMAIL extracted via regex separately.

In [19]:
import re
import tiktoken
from gliner import GLiNER

from ingest.chunker import chunk_markdown

model = GLiNER.from_pretrained("gliner-community/gliner_large-v2.5")

# Load the full document text
doc_path = src_dir / "converted_md" / "2026q1-alphabet-earnings-release.md"
second_doc_path = src_dir / "converted_md" / "Okobi3892026JAMMR164329.md"
full_text = second_doc_path.read_text(encoding="utf-8")
# full_text = doc_path.read_text(encoding="utf-8")

labels = ["person", "organization", "product", "location", "date", "concept"]

# --- Run GLiNER over fixed 3k-token windows instead of per markdown chunk ---
# Bigger windows give the model more context than a single ~1k-token chunk
# while staying well under a single call's limit. Windows are decoded straight
# from full_text's own tokens (not chunk_markdown's rejoined text), so their
# offsets track full_text almost exactly - unlike chunk.content, which
# collapses whitespace and needs the fuzzy locate_chunk_spans() match below.
WINDOW_TOKENS = 3000
enc = tiktoken.get_encoding("cl100k_base")
tokens = enc.encode(full_text)

raw_entities = []  # global-offset entities, before chunk assignment/dedup
cursor = 0
for i in range(0, len(tokens), WINDOW_TOKENS):
    window_text = enc.decode(tokens[i:i + WINDOW_TOKENS])
    found = model.predict_entities(window_text, labels, threshold=0.4)
    for e in found:
        raw_entities.append({
            "label": e["label"],
            "text": e["text"],
            "score": e["score"],
            "start": cursor + e["start"],
            "end": cursor + e["end"],
        })
    cursor += len(window_text)

num_windows = (len(tokens) + WINDOW_TOKENS - 1) // WINDOW_TOKENS
print(f"Ran GLiNER over {num_windows} window(s) of up to {WINDOW_TOKENS} tokens")
print(f"Raw entities found: {len(raw_entities)}")

# --- Email regex (no model needed) - folded into the same entity list ---
email_re = re.compile(r"[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}")
for m in email_re.finditer(full_text):
    raw_entities.append({
        "label": "email", "text": m.group(), "score": 1.0,
        "start": m.start(), "end": m.end(),
    })


def locate_chunk_spans(chunks: list, source_text: str) -> list[tuple[int, int]]:
    """Find each chunk's (start, end) character offsets in the source document.

    chunk_markdown() rebuilds each section's text by rejoining lines (via
    MarkdownHeaderTextSplitter), which collapses blank-line spacing - so
    chunk.content is rarely byte-identical to a substring of source_text even
    though the words are unchanged. Match on words instead of exact bytes: a
    regex that requires the same words in order, separated by any run of
    whitespace, finds the true offsets regardless of how much whitespace was
    collapsed.
    """
    spans = []
    cursor = 0
    for chunk in chunks:
        words = chunk.content.split()
        if not words:
            spans.append((cursor, cursor))
            continue
        pattern = re.compile(r"\s+".join(re.escape(w) for w in words))
        m = pattern.search(source_text, cursor) or pattern.search(source_text)
        if not m:
            # Couldn't locate even with whitespace tolerance - keep it a
            # zero-width span so it never matches an entity by position.
            spans.append((cursor, cursor))
            continue
        spans.append((m.start(), m.end()))
        cursor = m.end()
    return spans


def chunk_ids_for_span(span_start: int, span_end: int, spans: list[tuple[int, int]]) -> list[int]:
    """Every chunk id whose (start, end) range overlaps the given span."""
    return [
        idx for idx, (c_start, c_end) in enumerate(spans)
        if span_start < c_end and span_end > c_start
    ]


chunks = chunk_markdown(full_text)
chunk_spans = locate_chunk_spans(chunks, full_text)

# --- Assign each entity to its chunk id(s), dropping duplicates within a chunk ---
per_chunk_entities: list[list[dict]] = [[] for _ in chunks]
seen: set[tuple[int, str, str]] = set()  # (chunk_id, label, text_lower)
for e in sorted(raw_entities, key=lambda x: x["start"]):
    for chunk_id in chunk_ids_for_span(e["start"], e["end"], chunk_spans):
        key = (chunk_id, e["label"], e["text"].strip().lower())
        if key in seen:
            # Same entity text+label already seen in this chunk - drop it.
            continue
        seen.add(key)
        per_chunk_entities[chunk_id].append(e)

for idx, chunk in enumerate(chunks):
    ents = per_chunk_entities[idx]
    print(f"\nChunk {idx} ({chunk.tokens} tokens, breadcrumb: {chunk.breadcrumb()}):")
    if not ents:
        print("  (no entities)")
        continue
    for e in ents:
        print(f"  [{e['label']:12s}] {e['text']:40s} (score: {e['score']:.3f}, span: {e['start']}-{e['end']})")


Fetching 12 files: 100%|██████████| 12/12 [00:00<00:00, 1713.42it/s]


Ran GLiNER over 4 window(s) of up to 3000 tokens
Raw entities found: 7

Chunk 0 (92 tokens, breadcrumb: **_Journal of Advances in Medicine and Medical Research_** > tokens: 92):
  (no entities)

Chunk 1 (3780 tokens, breadcrumb: **Factors Associated with High Blood Cholesterol among Indigenous Adults in Canada: A Cross-sectional Analysis of the 2022 Canadian Community Health Survey** > **John Chibueze Ogbu**<sup>**a**</sup> **, Oluleke A. Adeyanju**<sup>**b**</sup> **, Kingdom Awuji**<sup>**c**</sup> **, Okelue Edwards Okobi**<sup>**d***</sup> **, Adetola Ifeoluwa Bakare**<sup>**e**</sup> **and Mofiyinfoluwa Fagbemi**<sup>**f**</sup> > tokens: 3780):
  [person      ] John Chibueze Ogbu                       (score: 0.693, span: 444-462)
  [person      ] Oluleke A. Adeyanju                      (score: 0.668, span: 485-504)
  [person      ] Kingdom Awuji                            (score: 0.680, span: 527-540)
  [person      ] Okelue Edwards Okobi                     (score: 0.662, span

In [ ]:
## BERT

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_ollama.llms import OllamaLLM

OLLAMA_HOST = "http://localhost:11434"
NER_MODEL= "qwen2.5:7b-instruct-q4_K_M"
NER_SYSTEM_PROMPT = """You are a named entity extractor. Extract entities from the provided text.
  Return ONLY a JSON object with these keys:
  - persons: list of person names
  - orgs: list of organization names
  - locations: list of geographic locations
  - dates: list of date/time expressions
  - concepts: list of technical terms, domain-specific terminology, key topics

  Rules:
  - Only extract entities explicitly present in the text
  - Do not infer or generate entities not in the text
  - Include entities found in tables, headers, and structured content
  - Return empty lists if no entities of that type are found"""

def qwen_ner(text: str) -> str:
    response = requests.post(
        f"{OLLAMA_HOST}/api",
        json={
            "model": NER_MODEL,
            "messages": [{"role": "user", "content": NER_SYSTEM_PROMPT + "\n\n" + text}],
        },
    )
    return response.json()

model = OllamaLLM(model="qwen2.5:7b-instruct-q4_K_M")



# request.post('http://localhost:11434/api', )

### Qwen 2.5 NER Extraction Test
Test NER on the Alphabet earnings document using Qwen 2.5 7B (q4 quantized).
Validates: full-document extraction, markdown table handling (GLiNER's blind spot),
minimal normalization, and entity-to-chunk assignment.

In [ ]:
import json
import re
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

# --- Load Qwen 2.5 7B (q4 quantized) ---
MODEL_NAME = "Qwen/Qwen2.5-7B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
print("Loading Qwen 2.5 model...")
qwen_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
)

NER_SYSTEM_PROMPT = """You are a named entity extractor. Extract entities from the provided text.
Return ONLY a JSON object with these keys:
- persons: list of person names
- orgs: list of organization names
- locations: list of geographic locations
- dates: list of date/time expressions
- concepts: list of technical terms, domain-specific terminology, key topics

Rules:
- Only extract entities explicitly present in the text
- Do not infer or generate entities not in the text
- Include entities found in tables, headers, and structured content
- Return empty lists if no entities of that type are found"""


def qwen_ner(text: str) -> dict:
    """Run NER via direct model inference."""
    messages = [
        {"role": "system", "content": NER_SYSTEM_PROMPT},
        {"role": "user", "content": text},
    ]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to(qwen_model.device)
    with torch.no_grad():
        outputs = qwen_model.generate(**inputs, max_new_tokens=2048, temperature=0.1, do_sample=True)
    response = tokenizer.decode(outputs[0][inputs.input_ids.shape[-1]:], skip_special_tokens=True)
    return json.loads(response)


# --- Contact regex ---
email_re = re.compile(r"[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}")
phone_re = re.compile(r"(?:\+?1[-.\s]?)?\(?\d{3}\)?[-.\s]?\d{3}[-.\s]?\d{4}")

# --- Load document ---
doc_path = src_dir / "converted_md" / "2026q1-alphabet-earnings-release.md"
full_text = doc_path.read_text(encoding="utf-8")

# --- Test 1: Full document NER ---
print("=== Test 1: Full document NER via Qwen 2.5 ===")
print(f"Document length: {len(full_text)} chars\n")

entities = qwen_ner(full_text)
for label, items in entities.items():
    print(f"  {label}: {items}")

emails = email_re.findall(full_text)
phones = phone_re.findall(full_text)
print(f"\n  contacts (regex): emails={emails}, phones={phones}")

# --- Test 2: Geography table section (GLiNER's blind spot) ---
print("\n=== Test 2: Geography table section (GLiNER failed here) ===")
geo_start = full_text.find("Revenues by Geography")
geo_section = full_text[geo_start:geo_start + 2000]
print(f"Section length: {len(geo_section)} chars\n")

geo_entities = qwen_ner(geo_section)
for label, items in geo_entities.items():
    print(f"  {label}: {items}")

# --- Test 3: Minimal normalization ---
print("\n=== Test 3: Normalized + deduplicated entities ===")

def normalize_entities(raw: dict) -> dict:
    result = {}
    for label, items in raw.items():
        seen = set()
        normed = []
        for item in items:
            clean = item.strip().lower()
            if label == "dates" and len(clean) < 6:
                continue
            if clean not in seen:
                seen.add(clean)
                normed.append(clean)
        result[label] = normed
    return result

normed = normalize_entities(entities)
for label, items in normed.items():
    print(f"  {label} ({len(items)}): {items}")

# --- Test 4: Entity-to-chunk assignment ---
print("\n=== Test 4: Entity-to-chunk assignment ===")
chunks = chunk_markdown(full_text)
contacts = list(set(emails + phones))

for idx, chunk in enumerate(chunks[:5]):
    chunk_lower = chunk.content.lower()
    assigned = {}
    for label, items in normed.items():
        matches = [e for e in items if e in chunk_lower]
        if matches:
            assigned[label] = matches
    contact_matches = [c for c in contacts if c.lower() in chunk_lower]
    if contact_matches:
        assigned["contacts"] = contact_matches

    print(f"\n  Chunk {idx} ({chunk.tokens} tokens, breadcrumb: {chunk.breadcrumb()}):")
    if assigned:
        for label, matches in assigned.items():
            print(f"    {label}: {', '.join(matches)}")
    else:
        print(f"    (no entities)")

In [ ]:

print(get_collection(c_client, "chunks"))